# Amenities Pipeline

This notebook computes amenity distances and names for each HDB resale flat address:
1. Distance and name of the nearest **hawker centre** open at the time of the transaction
2. Straight-line distance to the **CBD** (Raffles Place MRT proxy)
3. Distance to the nearest **MOE primary school** + list of school names within 1 km
4. Distance to the nearest **NParks-managed park** + list of park names within 1 km
5. Distance and name of the nearest **SportSG sport facility**
6. Distance and name of the nearest **shopping mall**
7. Distance and name of the nearest **polyclinic or hospital** (38 facilities)
8. **Real resale price** adjusted to Q4 2025 using the HDB Resale Price Index

All distances are computed from cached coordinates — no API calls are needed after the first run (except Phase 6 which geocodes schools on first run, and Phase 10 which geocodes healthcare facilities on first run).

## Inputs
- `../merged_data/hdb_with_train_distances.csv` — enriched HDB dataset (contains `lat`, `lon`, `nearest_train_name` from `train_pipeline.ipynb`)
- `../data/geocode_cache.json` — geocoded lat/lon per address (reference)
- `../data/HawkerCentresGEOJSON.geojson` — hawker centre locations and status metadata
- `../data/Generalinformationofschools.csv` — MOE school directory (Phase 6)
- `../data/Parks.geojson` — NParks managed green spaces (Phase 7)
- `../data/SportSGSportFacilitiesGEOJSON.geojson` — SportSG sport facilities (Phase 8)
- `../data/shoppingmalls.csv` — shopping mall locations (Phase 9)
- `../data/healthcare_address.csv` — polyclinics and hospitals (Phase 10)

## How to Run
Run cells top to bottom. Phases 3–10 are fully vectorised and run in seconds. Phase 6 geocodes primary schools via the OneMap API on first run (~60 s for ~179 schools); Phase 10 geocodes 38 healthcare facilities on first run (~15 s). Subsequent runs use the cache and complete instantly.

## Output Files
| File | Description |
|------|-------------|
| `../merged_data/hdb_with_amenities_macro.csv` | Full dataset with all amenity distance/name columns and real resale price |

### Columns added by this notebook
| Column | Description |
|--------|-------------|
| `dist_nearest_hawker_m` | Distance to nearest hawker centre open at transaction date (m) |
| `nearest_hawker_name` | Name of the nearest hawker centre |
| `dist_cbd_m` | Straight-line distance to Raffles Place MRT / CBD (m) |
| `dist_nearest_primary_m` | Distance to nearest MOE primary school (m) |
| `primary_schools_1km` | Pipe-separated names of primary schools within 1 km (empty string if none) |
| `dist_nearest_park_m` | Distance to nearest NParks-managed park (m) |
| `parks_1km` | Pipe-separated names of parks within 1 km (empty string if none) |
| `dist_nearest_sportsg_m` | Distance to nearest SportSG sport facility (m) |
| `nearest_sportsg_name` | Name of the nearest SportSG facility |
| `dist_nearest_mall_m` | Distance to nearest shopping mall (m) |
| `nearest_mall_name` | Name of the nearest shopping mall |
| `dist_nearest_healthcare_m` | Distance to nearest polyclinic or hospital (m) |
| `nearest_healthcare_name` | Name of the nearest polyclinic or hospital |
| `resale_price_real` | Resale price adjusted to Q4 2025 RPI = 203.6 |

## Hawker Centre Status Rules
| STATUS | Treatment |
|--------|-----------|
| `Existing` | Always open (all predate 2017; dates are year-only strings) |
| `Existing (replacement)` | Always open (location had a hawker centre before the rebuild) |
| `Existing (new)` | Open only if `EST_ORIGINAL_COMPLETION_DATE ≤ transaction month` |
| `Interim Centre` | Open only if `EST_ORIGINAL_COMPLETION_DATE ≤ transaction month` |
| `Under Construction` | Always excluded |

In [9]:
import pandas as pd
import numpy as np
import json
import os

print("Libraries imported successfully!")

Libraries imported successfully!


## Phase 1 — Load Data

Load the enriched HDB dataset (which already contains geocoded `lat`/`lon` from `train_pipeline.ipynb`) and the geocode cache for reference.

In [10]:
GEOCODE_CACHE_FILE = '../../data/geocode_cache.json'
INPUT_CSV          = '../../merged_data/hdb_with_train_distances.csv'

with open(GEOCODE_CACHE_FILE) as f:
    geocode_cache = json.load(f)
print(f"Geocode cache: {len(geocode_cache):,} addresses")

df = pd.read_csv(INPUT_CSV)
df['month'] = pd.to_datetime(df['month'])
print(f"Dataset shape: {df.shape}")
print(f"Date range:    {df['month'].min().strftime('%Y-%m')} to {df['month'].max().strftime('%Y-%m')}")
print(f"Rows with lat/lon: {df['lat'].notna().sum():,}")
print()
print(df[['month', 'block', 'street_name', 'lat', 'lon']].head(5))

Geocode cache: 9,685 addresses
Dataset shape: (140537, 22)
Date range:    2021-01 to 2026-03
Rows with lat/lon: 140,447

       month block        street_name       lat         lon
0 2021-01-01   170   ANG MO KIO AVE 4  1.374001  103.836432
1 2021-01-01   170   ANG MO KIO AVE 4  1.374001  103.836432
2 2021-01-01   331   ANG MO KIO AVE 1  1.362111  103.850766
3 2021-01-01   534  ANG MO KIO AVE 10  1.374058  103.854168
4 2021-01-01   561  ANG MO KIO AVE 10  1.370578  103.857855


## Phase 2 — Load Hawker Centre Data

Parse the GeoJSON into a flat DataFrame. For `Existing (new)` and `Interim Centre` entries, parse `EST_ORIGINAL_COMPLETION_DATE` with `dayfirst=True` to handle the `DD/M/YYYY` format. Parse failures (e.g. year-only strings like `"1983"` for `Existing` entries) produce `NaT` and are treated as always-open. Pre-compute the set of open hawker centres for every unique transaction month in the dataset.

In [11]:
HAWKER_GEOJSON = '../../data/HawkerCentresGEOJSON.geojson'

with open(HAWKER_GEOJSON) as f:
    geojson = json.load(f)

rows = []
for feat in geojson['features']:
    props = feat['properties']
    lon, lat = feat['geometry']['coordinates']   # GeoJSON standard: [lon, lat]
    date_str = props.get('EST_ORIGINAL_COMPLETION_DATE', '')
    est_dt   = pd.to_datetime(date_str, dayfirst=True, errors='coerce')
    rows.append({
        'name':              props.get('NAME', ''),
        'status':            props.get('STATUS', ''),
        'est_completion_dt': est_dt,
        'lat': lat,
        'lon': lon,
    })
hawker_df = pd.DataFrame(rows)

print(f"Total hawker centres loaded: {len(hawker_df):,}")
print("\nBy STATUS:")
print(hawker_df['status'].value_counts().to_string())

date_dep = hawker_df.query("status in ['Existing (new)', 'Interim Centre']").copy()
print(f"\nDate-dependent centres ({len(date_dep)}) — filtered by EST_ORIGINAL_COMPLETION_DATE:")
print(date_dep[['name', 'status', 'est_completion_dt']].to_string())

Total hawker centres loaded: 129

By STATUS:
status
Existing                  103
Existing (new)             16
Under Construction          6
Existing (replacement)      3
Interim Centre              1

Date-dependent centres (17) — filtered by EST_ORIGINAL_COMPLETION_DATE:
                                             name          status est_completion_dt
2                     Punggol Coast Hawker Centre  Existing (new)        2024-08-21
5    Bukit Timah Interim Hawker Centre and Market  Interim Centre        2024-10-01
11                      Jurong West Hawker Centre  Existing (new)        2017-07-28
15                      One Punggol Hawker Centre  Existing (new)        2022-04-29
17                   Bukit Canberra Hawker Centre  Existing (new)        2022-09-27
23                      Yishun Park Hawker Centre  Existing (new)        2017-07-03
32                 Bukit Batok West Hawker Centre  Existing (new)        2024-10-08
33                Fernvale Hawker Centre & Market  Ex

In [12]:
def is_hawker_open(status, est_dt, transaction_month):
    """Return True if the hawker centre was open at transaction_month."""
    if status == 'Under Construction':
        return False
    if status in ('Existing', 'Existing (replacement)'):
        return True          # always open; predates dataset or rebuilt on same site
    # 'Existing (new)' or 'Interim Centre': date-dependent
    if pd.isna(est_dt):
        return True          # parse failure → safe fallback (treat as always open)
    return transaction_month >= est_dt


def hawker_dists_vectorized(lats_arr, lons_arr, hawker_coords):
    """Compute minimum haversine distance (metres) from N addresses to H hawker centres.

    Parameters
    ----------
    lats_arr, lons_arr : 1-D float arrays, shape (N,)
    hawker_coords      : 2-D float array, shape (H, 2), columns [lat, lon]

    Returns
    -------
    1-D float array, shape (N,) — NaN for rows where lat/lon is NaN.
    """
    R = 6_371_000.0
    if len(hawker_coords) == 0:
        return np.full(len(lats_arr), np.nan)

    valid  = ~(np.isnan(lats_arr) | np.isnan(lons_arr))
    result = np.full(len(lats_arr), np.nan)
    if not valid.any():
        return result

    lat1 = np.radians(lats_arr[valid])[:, np.newaxis]       # (N_v, 1)
    lon1 = np.radians(lons_arr[valid])[:, np.newaxis]
    lat2 = np.radians(hawker_coords[:, 0])[np.newaxis, :]   # (1, H)
    lon2 = np.radians(hawker_coords[:, 1])[np.newaxis, :]

    a      = (np.sin((lat2 - lat1) / 2) ** 2
               + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    dist_m = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))  # (N_v, H)
    result[valid] = np.round(dist_m.min(axis=1), 1)
    return result


def nearest_with_name_vectorized(lats_arr, lons_arr, facility_coords, facility_names=None):
    """Compute distance and optionally the name of the nearest facility.

    Parameters
    ----------
    lats_arr, lons_arr : 1-D float arrays, shape (N,)
    facility_coords    : 2-D float array, shape (F, 2), columns [lat, lon]
    facility_names     : 1-D array-like of strings, shape (F,), or None

    Returns
    -------
    distances  : 1-D float array, shape (N,) — NaN for rows with null lat/lon
    names_out  : 1-D object array, shape (N,) — None where distance is NaN;
                 None everywhere if facility_names not provided
    """
    R = 6_371_000.0
    n = len(lats_arr)
    distances = np.full(n, np.nan)
    names_out = np.full(n, None, dtype=object)

    if len(facility_coords) == 0:
        return distances, names_out

    valid = ~(np.isnan(lats_arr) | np.isnan(lons_arr))
    if not valid.any():
        return distances, names_out

    lat1 = np.radians(lats_arr[valid])[:, np.newaxis]
    lon1 = np.radians(lons_arr[valid])[:, np.newaxis]
    lat2 = np.radians(facility_coords[:, 0])[np.newaxis, :]
    lon2 = np.radians(facility_coords[:, 1])[np.newaxis, :]

    a      = (np.sin((lat2 - lat1) / 2) ** 2
               + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    dist_m = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))  # (N_v, F)

    min_idx = np.argmin(dist_m, axis=1)
    distances[valid] = np.round(dist_m[np.arange(valid.sum()), min_idx], 1)

    if facility_names is not None:
        names_arr = np.asarray(facility_names)
        names_out[valid] = names_arr[min_idx]

    return distances, names_out


# Pre-compute open-hawker coordinate and name arrays for every unique month in the dataset.
print("Pre-computing open hawker sets per unique month...")
unique_months = sorted(df['month'].unique())
open_hawkers_by_month      = {}
open_hawker_names_by_month = {}
for month in unique_months:
    mask = hawker_df.apply(
        lambda r: is_hawker_open(r['status'], r['est_completion_dt'], month), axis=1)
    open_hawkers_by_month[month]      = hawker_df.loc[mask, ['lat', 'lon']].values
    open_hawker_names_by_month[month] = hawker_df.loc[mask, 'name'].values

# Spot-check: open counts at key dates
for m_str in ['2018-04-01', '2021-01-01', '2024-07-01']:
    m = pd.Timestamp(m_str)
    if m in open_hawkers_by_month:
        print(f"  {m_str[:7]}: {len(open_hawkers_by_month[m]):>3} hawker centres open")
print(f"Pre-computation complete for {len(unique_months)} unique months.")

Pre-computing open hawker sets per unique month...
  2021-01: 113 hawker centres open
  2024-07: 120 hawker centres open
Pre-computation complete for 63 unique months.


## Phase 3 — Compute Nearest Hawker Distance

For each transaction row, look up the set of hawker centres open at that transaction month, then compute the haversine distance to all of them and take the minimum. The computation is fully vectorised using NumPy broadcasting — no API calls are made. All rows sharing the same month are processed together in a single matrix operation.

Rows are processed by month group using vectorised NumPy — typically completes in a few seconds.

In [13]:
print("Computing nearest open hawker distance and name for all rows (grouped by month)...")
dist_col = np.full(len(df), np.nan)
name_col = np.full(len(df), None, dtype=object)

for month, group_idx in df.groupby('month').groups.items():
    coords = open_hawkers_by_month.get(month, np.zeros((0, 2)))
    names  = open_hawker_names_by_month.get(month, np.array([], dtype=str))
    lats   = df.loc[group_idx, 'lat'].values.astype(float)
    lons   = df.loc[group_idx, 'lon'].values.astype(float)
    dists, nearest_names = nearest_with_name_vectorized(lats, lons, coords, names)
    dist_col[group_idx] = dists
    name_col[group_idx] = nearest_names

df['dist_nearest_hawker_m'] = dist_col
df['nearest_hawker_name']   = name_col
non_null = int((~np.isnan(dist_col)).sum())
print(f"Done. Non-null values: {non_null:,} / {len(df):,}")

Computing nearest open hawker distance and name for all rows (grouped by month)...
Done. Non-null values: 140,447 / 140,537


## Phase 4 — Save Full Dataset

Print a summary of the nearest hawker distance column — null counts, statistics, and a town-level sanity check. No file is written here; all phases accumulate columns in memory and the dataset is saved once after all amenity phases and the RPI price adjustment are complete.

In [14]:
has_dist = df['dist_nearest_hawker_m'].notna().sum()
missing  = df['dist_nearest_hawker_m'].isna().sum()

print(f"\n{'='*60}")
print("SUMMARY — dist_nearest_hawker_m")
print(f"{'='*60}")
print(f"Non-null: {has_dist:,}   Null (failed geocode): {missing:,}")
print(f"  Min:  {df['dist_nearest_hawker_m'].min():.1f} m")
print(f"  Mean: {df['dist_nearest_hawker_m'].mean():.1f} m")
print(f"  Max:  {df['dist_nearest_hawker_m'].max():.1f} m")

town_hk = (df.groupby('town')['dist_nearest_hawker_m']
             .mean().sort_values()
             .rename('mean_dist_hawker_m').round(0).astype(int))
print(f"\nTop 5 towns closest to a hawker centre:")
print(town_hk.head(5).to_string())
print(f"\nTop 5 towns furthest from a hawker centre:")
print(town_hk.tail(5).to_string())


SUMMARY — dist_nearest_hawker_m
Non-null: 140,447   Null (failed geocode): 90
  Min:  0.0 m
  Mean: 949.3 m
  Max:  4889.0 m

Top 5 towns closest to a hawker centre:
town
CENTRAL AREA     195
BUKIT MERAH      306
MARINE PARADE    307
ANG MO KIO       316
GEYLANG          335

Top 5 towns furthest from a hawker centre:
town
BUKIT TIMAH      1360
SENGKANG         1361
BUKIT BATOK      1450
PUNGGOL          2035
CHOA CHU KANG    2470


## Phase 5 — CBD Distance

Compute the straight-line distance from each flat to the Central Business District (CBD), using Raffles Place MRT as the proxy — consistent with Singapore hedonic pricing literature. No API calls are needed; the computation reuses `hawker_dists_vectorized()` with CBD as the sole target point.

In [15]:
# Raffles Place MRT used as CBD proxy, consistent with Singapore hedonic pricing literature
CBD_LAT = 1.2830
CBD_LON = 103.8513

# Reuse hawker_dists_vectorized with CBD as the sole target point.
# Returns NaN for rows with null lat/lon — consistent with dist_nearest_hawker_m behaviour.
cbd_coords = np.array([[CBD_LAT, CBD_LON]])
df['dist_cbd_m'] = hawker_dists_vectorized(
    df['lat'].values.astype(float),
    df['lon'].values.astype(float),
    cbd_coords
)

# ── Summary ──────────────────────────────────────────────────────────────────
non_null = df['dist_cbd_m'].notna().sum()
null_cnt = df['dist_cbd_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY — dist_cbd_m")
print(f"{'='*60}")
print(f"Non-null: {non_null:,}   Null (failed geocode): {null_cnt:,}")
print(f"  Min:  {df['dist_cbd_m'].min():,.1f} m")
print(f"  Mean: {df['dist_cbd_m'].mean():,.1f} m")
print(f"  Max:  {df['dist_cbd_m'].max():,.1f} m")

# ── Sanity check: mean CBD distance by town ──────────────────────────────────
town_cbd = (df.groupby('town')['dist_cbd_m']
              .mean()
              .sort_values()
              .rename('mean_dist_cbd_m')
              .round(0)
              .astype(int))
print(f"\nTop 5 closest towns to CBD:")
print(town_cbd.head(5).to_string())
print(f"\nTop 5 furthest towns from CBD:")
print(town_cbd.tail(5).to_string())


SUMMARY — dist_cbd_m
Non-null: 140,447   Null (failed geocode): 90
  Min:  591.8 m
  Mean: 12,533.2 m
  Max:  20,312.7 m

Top 5 closest towns to CBD:
town
CENTRAL AREA       1685
BUKIT MERAH        3431
KALLANG/WHAMPOA    4327
TOA PAYOH          6031
GEYLANG            6158

Top 5 furthest towns from CBD:
town
YISHUN           15938
CHOA CHU KANG    16278
JURONG WEST      17212
WOODLANDS        18526
SEMBAWANG        18741


## Phase 6 — Primary School Distance

Geocode all MOE primary schools by postal code using the OneMap search API, then compute the straight-line distance from each HDB flat to the nearest primary school. School coordinates are cached in `data/school_geocode_cache.json` — re-running this cell skips schools that are already cached.

In [16]:
import requests
import time
from dotenv import load_dotenv

# ── Load credentials (needed for OneMap search API) ──────────────────────────
load_dotenv('../../.env')
ONEMAP_EMAIL    = os.getenv('ONEMAP_EMAIL')
ONEMAP_PASSWORD = os.getenv('ONEMAP_PASSWORD')

_token   = {"access_token": None, "expiry": 0}
_session = requests.Session()

def _get_token():
    if _token["access_token"] and time.time() < int(_token["expiry"]) - 60:
        return _token["access_token"]
    resp = _session.post(
        "https://www.onemap.gov.sg/api/auth/post/getToken",
        json={"email": ONEMAP_EMAIL, "password": ONEMAP_PASSWORD}, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    _token["access_token"] = data["access_token"]
    _token["expiry"] = int(data["expiry_timestamp"])
    return _token["access_token"]

# ── Load and filter MOE school directory ─────────────────────────────────────
SCHOOLS_CSV = '../../data/Generalinformationofschools.csv'
schools_raw = pd.read_csv(SCHOOLS_CSV)
schools = (schools_raw[schools_raw['mainlevel_code'] == 'PRIMARY']
             [['school_name', 'address', 'postal_code']]
             .copy()
             .reset_index(drop=True))
print(f"Primary schools loaded: {len(schools)}")

# ── Geocode by postal code ────────────────────────────────────────────────────
SCHOOL_GEOCODE_CACHE = '../../data/school_geocode_cache.json'
if os.path.exists(SCHOOL_GEOCODE_CACHE):
    with open(SCHOOL_GEOCODE_CACHE) as f:
        school_cache = json.load(f)
    print(f"Loaded existing cache: {len(school_cache)} postal codes")
else:
    school_cache = {}
    print("No existing cache — starting fresh.")

GEOCODE_URL = 'https://www.onemap.gov.sg/api/common/elastic/search'
new_calls   = 0

for i, row in schools.iterrows():
    postal = str(row['postal_code']).zfill(6)
    if postal in school_cache:
        continue                             # already cached — skip API call
    try:
        resp = _session.get(
            GEOCODE_URL,
            params={'searchVal': postal, 'returnGeom': 'Y',
                    'getAddrDetails': 'Y', 'pageNum': 1},
            headers={'Authorization': f'Bearer {_get_token()}'},
            timeout=10)
        resp.raise_for_status()
        results = resp.json().get('results', [])
        if results:
            school_cache[postal] = {
                'lat': float(results[0]['LATITUDE']),
                'lon': float(results[0]['LONGITUDE']),
            }
        else:
            print(f"  WARNING: no results for {row['school_name']} (postal {postal})")
            school_cache[postal] = None
    except requests.RequestException as e:
        print(f"  ERROR geocoding {row['school_name']}: {e}")
        school_cache[postal] = None
    new_calls += 1
    time.sleep(0.3)
    if new_calls % 10 == 0:                 # checkpoint every 10 new calls
        with open(SCHOOL_GEOCODE_CACHE, 'w') as f:
            json.dump(school_cache, f)
        print(f"  Checkpoint saved ({new_calls} new API calls so far...)")

# Final save
with open(SCHOOL_GEOCODE_CACHE, 'w') as f:
    json.dump(school_cache, f)

ok  = sum(1 for v in school_cache.values() if v is not None)
bad = sum(1 for v in school_cache.values() if v is None)
print(f"\nGeocoding complete: {ok} succeeded, {bad} failed")
print(f"Cache saved to: {SCHOOL_GEOCODE_CACHE}")

Primary schools loaded: 179
Loaded existing cache: 179 postal codes

Geocoding complete: 179 succeeded, 0 failed
Cache saved to: ../../data/school_geocode_cache.json


In [18]:
# Build parallel arrays of school coordinates and names for geocoded schools
school_rows_geocoded = []
for _, srow in schools.iterrows():
    postal = str(srow['postal_code']).zfill(6)
    cached = school_cache.get(postal)
    if cached is not None:
        school_rows_geocoded.append({
            'name': srow['school_name'],
            'lat':  cached['lat'],
            'lon':  cached['lon'],
        })
school_geocoded_df = pd.DataFrame(school_rows_geocoded)
school_coords      = school_geocoded_df[['lat', 'lon']].values.astype(float)
school_names_arr   = school_geocoded_df['name'].values
print(f"Using {len(school_coords)} geocoded primary schools for distance computation.")

# Nearest primary school distance
df['dist_nearest_primary_m'], _ = nearest_with_name_vectorized(
    df['lat'].values.astype(float),
    df['lon'].values.astype(float),
    school_coords
)

# Primary schools within 1 km — full distance matrix (N_valid × n_schools)
R = 6_371_000.0
flat_lats = df['lat'].values.astype(float)
flat_lons = df['lon'].values.astype(float)
valid     = ~(np.isnan(flat_lats) | np.isnan(flat_lons))

lat1 = np.radians(flat_lats[valid])[:, np.newaxis]
lon1 = np.radians(flat_lons[valid])[:, np.newaxis]
lat2 = np.radians(school_coords[:, 0])[np.newaxis, :]
lon2 = np.radians(school_coords[:, 1])[np.newaxis, :]
a      = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
dist_m = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))   # (N_valid, n_schools)

within_mask = dist_m <= 1000
schools_1km = np.full(len(df), '', dtype=object)
schools_1km[valid] = ['|'.join(school_names_arr[within_mask[i]]) for i in range(valid.sum())]
df['primary_schools_1km'] = schools_1km

# ── Summary ──────────────────────────────────────────────────────────────────
non_null = df['dist_nearest_primary_m'].notna().sum()
null_cnt = df['dist_nearest_primary_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY — dist_nearest_primary_m / primary_schools_1km")
print(f"{'='*60}")
print(f"Non-null: {non_null:,}   Null (failed geocode): {null_cnt:,}")
print(f"  Min:  {df['dist_nearest_primary_m'].min():,.1f} m")
print(f"  Mean: {df['dist_nearest_primary_m'].mean():,.1f} m")
print(f"  Max:  {df['dist_nearest_primary_m'].max():,.1f} m")
n_with_schools = (df['primary_schools_1km'] != '').sum()
print(f"\nprimary_schools_1km: {n_with_schools:,} rows have ≥1 school within 1 km")

# ── Sanity check: mean primary school distance by town ───────────────────────
town_sch = (df.groupby('town')['dist_nearest_primary_m']
              .mean()
              .sort_values()
              .rename('mean_dist_primary_m')
              .round(0)
              .astype(int))
print(f"\nTop 5 towns with closest primary schools:")
print(town_sch.head(5).to_string())
print(f"\nTop 5 towns with furthest primary schools:")
print(town_sch.tail(5).to_string())

# Count of primary schools within 1 km
df['num_primary_1km'] = df['primary_schools_1km'].apply(
    lambda x: len(x.split('|')) if x else 0)
print(f"num_primary_1km — mean: {df['num_primary_1km'].mean():.2f}, max: {df['num_primary_1km'].max()}")

Using 179 geocoded primary schools for distance computation.

SUMMARY — dist_nearest_primary_m / primary_schools_1km
Non-null: 140,447   Null (failed geocode): 90
  Min:  43.8 m
  Mean: 426.6 m
  Max:  3,296.6 m

primary_schools_1km: 135,303 rows have ≥1 school within 1 km

Top 5 towns with closest primary schools:
town
MARINE PARADE    273
PUNGGOL          291
SENGKANG         311
BUKIT PANJANG    343
BUKIT TIMAH      351

Top 5 towns with furthest primary schools:
town
CLEMENTI           590
QUEENSTOWN         603
KALLANG/WHAMPOA    606
BISHAN             689
JURONG EAST        712
num_primary_1km — mean: 3.01, max: 9


## Phase 7 — Parks Distance

Load NParks managed green spaces from `data/Parks.geojson`, exclude non-park features (playgrounds, car parks, terminals, etc.), then compute the straight-line distance from each HDB flat to the nearest retained park. No API calls needed — all computation is local.

In [19]:
PARKS_GEOJSON = '../../data/Parks.geojson'

with open(PARKS_GEOJSON) as f:
    parks_geojson = json.load(f)

rows = []
for feat in parks_geojson['features']:
    props = feat['properties']
    lon, lat = feat['geometry']['coordinates']   # GeoJSON standard: [lon, lat]
    rows.append({'name': props.get('NAME', ''), 'lat': lat, 'lon': lon})
parks_df = pd.DataFrame(rows)
print(f"Total features loaded: {len(parks_df)}")

# Exclusion rules — applied to uppercase NAME field
name = parks_df['name']
rules = {
    'ends with PLAYGROUND':    name.str.endswith('PLAYGROUND',    na=False),
    'ends with TERMINAL':      name.str.endswith('TERMINAL',      na=False),
    'ends with NURSERY':       name.str.endswith('NURSERY',       na=False),
    'ends with STATELAND':     name.str.endswith('STATELAND',     na=False),
    'ends with LINKWAY':       name.str.endswith('LINKWAY',       na=False),
    'contains CAR PARK':       name.str.contains('CAR PARK',      na=False),
    'contains FOOTBALL FIELD': name.str.contains('FOOTBALL FIELD', na=False),
    'contains SPORTSG':        name.str.contains('SPORTSG',       na=False),
}
for label, mask in rules.items():
    print(f"  Excluded ({label}): {mask.sum()}")

excl_mask = pd.concat(rules.values(), axis=1).any(axis=1)
parks_df  = parks_df[~excl_mask].reset_index(drop=True)
print(f"Parks after filtering: {len(parks_df)}")

Total features loaded: 450
  Excluded (ends with PLAYGROUND): 24
  Excluded (ends with TERMINAL): 1
  Excluded (ends with NURSERY): 1
  Excluded (ends with STATELAND): 1
  Excluded (ends with LINKWAY): 1
  Excluded (contains CAR PARK): 1
  Excluded (contains FOOTBALL FIELD): 1
  Excluded (contains SPORTSG): 1
Parks after filtering: 419


In [21]:
park_coords    = parks_df[['lat', 'lon']].values.astype(float)
park_names_arr = parks_df['name'].values
print(f"Using {len(park_coords)} parks for distance computation.")

# Nearest park distance
df['dist_nearest_park_m'], _ = nearest_with_name_vectorized(
    df['lat'].values.astype(float),
    df['lon'].values.astype(float),
    park_coords
)

# Parks within 1 km — full distance matrix (N_valid × n_parks)
R = 6_371_000.0
flat_lats = df['lat'].values.astype(float)
flat_lons = df['lon'].values.astype(float)
valid     = ~(np.isnan(flat_lats) | np.isnan(flat_lons))

lat1 = np.radians(flat_lats[valid])[:, np.newaxis]
lon1 = np.radians(flat_lons[valid])[:, np.newaxis]
lat2 = np.radians(park_coords[:, 0])[np.newaxis, :]
lon2 = np.radians(park_coords[:, 1])[np.newaxis, :]
a      = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
dist_m = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))   # (N_valid, n_parks)

within_mask = dist_m <= 1000
parks_1km = np.full(len(df), '', dtype=object)
parks_1km[valid] = ['|'.join(park_names_arr[within_mask[i]]) for i in range(valid.sum())]
df['parks_1km'] = parks_1km

non_null = df['dist_nearest_park_m'].notna().sum()
null_cnt = df['dist_nearest_park_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY — dist_nearest_park_m / parks_1km")
print(f"{'='*60}")
print(f"Non-null: {non_null:,}   Null (failed geocode): {null_cnt:,}")
print(f"  Min:  {df['dist_nearest_park_m'].min():,.1f} m")
print(f"  Mean: {df['dist_nearest_park_m'].mean():,.1f} m")
print(f"  Max:  {df['dist_nearest_park_m'].max():,.1f} m")
n_with_parks = (df['parks_1km'] != '').sum()
print(f"\nparks_1km: {n_with_parks:,} rows have ≥1 park within 1 km")

town_pk = (df.groupby('town')['dist_nearest_park_m']
             .mean().sort_values()
             .rename('mean_dist_park_m').round(0).astype(int))
print(f"\nTop 5 towns closest to a park:")
print(town_pk.head(5).to_string())
print(f"\nTop 5 towns furthest from a park:")
print(town_pk.tail(5).to_string())

# Count of parks within 1 km
df['num_parks_1km'] = df['parks_1km'].apply(
    lambda x: len(x.split('|')) if x else 0)
print(f"num_parks_1km — mean: {df['num_parks_1km'].mean():.2f}, max: {df['num_parks_1km'].max()}")

Using 419 parks for distance computation.

SUMMARY — dist_nearest_park_m / parks_1km
Non-null: 140,447   Null (failed geocode): 90
  Min:  44.3 m
  Mean: 763.8 m
  Max:  2,217.4 m

parks_1km: 105,963 rows have ≥1 park within 1 km

Top 5 towns closest to a park:
town
CENTRAL AREA     303
BUKIT TIMAH      371
MARINE PARADE    396
BISHAN           438
GEYLANG          449

Top 5 towns furthest from a park:
town
PUNGGOL           944
BUKIT PANJANG    1014
JURONG EAST      1149
WOODLANDS        1216
BUKIT BATOK      1259
num_parks_1km — mean: 2.31, max: 18


## Phase 8 — SportSG Sport Facility Distance

Load SportSG managed sport facilities from `data/SportSGSportFacilitiesGEOJSON.geojson` and compute the straight-line distance from each HDB flat to the nearest facility. No exclusion filtering is needed — all 45 entries are legitimate sport facilities. No API calls required.

In [22]:
SPORTSG_GEOJSON = '../../data/SportSGSportFacilitiesGEOJSON.geojson'

with open(SPORTSG_GEOJSON) as f:
    sportsg_geojson = json.load(f)

rows = []
for feat in sportsg_geojson['features']:
    props = feat['properties']
    lon, lat = feat['geometry']['coordinates']   # GeoJSON standard: [lon, lat]
    rows.append({'venue': props.get('VENUE', ''), 'lat': lat, 'lon': lon})
sportsg_df = pd.DataFrame(rows)

print(f"Total SportSG facilities loaded: {len(sportsg_df)}")
print("\nVenue list:")
for _, row in sportsg_df.iterrows():
    print(f"  {row['venue']}")

sportsg_coords    = sportsg_df[['lat', 'lon']].values.astype(float)
sportsg_names_arr = sportsg_df['venue'].values

df['dist_nearest_sportsg_m'], nearest_sportsg = nearest_with_name_vectorized(
    df['lat'].values.astype(float),
    df['lon'].values.astype(float),
    sportsg_coords,
    sportsg_names_arr
)
df['nearest_sportsg_name'] = nearest_sportsg

non_null = df['dist_nearest_sportsg_m'].notna().sum()
null_cnt = df['dist_nearest_sportsg_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY — dist_nearest_sportsg_m")
print(f"{'='*60}")
print(f"Non-null: {non_null:,}   Null (failed geocode): {null_cnt:,}")
print(f"  Min:  {df['dist_nearest_sportsg_m'].min():,.1f} m")
print(f"  Mean: {df['dist_nearest_sportsg_m'].mean():,.1f} m")
print(f"  Max:  {df['dist_nearest_sportsg_m'].max():,.1f} m")

town_sg = (df.groupby('town')['dist_nearest_sportsg_m']
             .mean().sort_values()
             .rename('mean_dist_sportsg_m').round(0).astype(int))
print(f"\nTop 5 towns closest to a SportSG facility:")
print(town_sg.head(5).to_string())
print(f"\nTop 5 towns furthest from a SportSG facility:")
print(town_sg.tail(5).to_string())

Total SportSG facilities loaded: 45

Venue list:
  Bukit Canberra Sport Centre
  Bukit Gombak Sport Centre
  Burghley Squash & Tennis Centre
  Choa Chu Kang Sport Centre
  Clementi Sport Centre
  Clementi Stadium
  Delta Sport Centre
  Geylang East Swimming Complex
  Heartbeat@Bedok ActiveSG Sport Centre
  Hougang Sport Centre
  Jalan Besar Sport Centre
  Jurong East Sport Centre
  Jurong Lake Gardens ActiveSG Gym & Pool
  Jurong West Sport Centre
  Kallang Basin Swimming Complex
  ActiveSG Sport Park @ Bedok North
  ActiveSG Gym@Ang Mo Kio Community Centre
  ActiveSG Gym@Enabling Village
  ActiveSG Gym@Fernvale Square
  ActiveSG Gym@Serangoon Central
  ActiveSG Gym@Toa Payoh
  ActiveSG Gym@Toa Payoh West Community Centre
  ActiveSG Hockey Village @ Boon Lay
  ActiveSG Sport Village@Jurong Town
  Ang Mo Kio Swimming Complex
  Bedok Stadium
  Bishan Sport Centre
  Bishan Clubhouse
  Bukit Batok Swimming Complex
  Kallang Sport Centre
  Katong Swimming Complex
  MOE (Evans)
  Tampines Sp

## Phase 9 — Shopping Mall Distance

Load shopping malls from `data/shoppingmalls.csv` (238 rows). Deduplicate by mall name — multi-entrance malls and data-entry duplicates are collapsed by taking the mean lat/lon per name group, leaving 221 unique mall locations. Compute straight-line distance from each HDB flat to the nearest mall using `hawker_dists_vectorized()`. No API calls required.

In [23]:
MALLS_CSV = '../../data/shoppingmalls.csv'
malls_raw = pd.read_csv(MALLS_CSV)
print(f"Total rows loaded: {len(malls_raw)}")

# Deduplicate by name — multi-entrance malls and data-entry duplicates
malls_df = (malls_raw.groupby('name', as_index=False)
              .agg(lat=('lat', 'mean'), lon=('lon', 'mean')))
n_dupes = len(malls_raw) - len(malls_df)
print(f"Unique malls after deduplication: {len(malls_df)}")
print(f"Duplicate rows collapsed: {n_dupes}")

mall_coords    = malls_df[['lat', 'lon']].values.astype(float)
mall_names_arr = malls_df['name'].values

df['dist_nearest_mall_m'], nearest_mall = nearest_with_name_vectorized(
    df['lat'].values.astype(float),
    df['lon'].values.astype(float),
    mall_coords,
    mall_names_arr
)
df['nearest_mall_name'] = nearest_mall

non_null = df['dist_nearest_mall_m'].notna().sum()
null_cnt = df['dist_nearest_mall_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY — dist_nearest_mall_m")
print(f"{'='*60}")
print(f"Non-null: {non_null:,}   Null (failed geocode): {null_cnt:,}")
print(f"  Min:  {df['dist_nearest_mall_m'].min():,.1f} m")
print(f"  Mean: {df['dist_nearest_mall_m'].mean():,.1f} m")
print(f"  Max:  {df['dist_nearest_mall_m'].max():,.1f} m")

town_mall = (df.groupby('town')['dist_nearest_mall_m']
               .mean().sort_values()
               .rename('mean_dist_mall_m').round(0).astype(int))
print(f"\nTop 5 towns closest to a shopping mall:")
print(town_mall.head(5).to_string())
print(f"\nTop 5 towns furthest from a shopping mall:")
print(town_mall.tail(5).to_string())

Total rows loaded: 238
Unique malls after deduplication: 221
Duplicate rows collapsed: 17

SUMMARY — dist_nearest_mall_m
Non-null: 140,447   Null (failed geocode): 90
  Min:  5.9 m
  Mean: 641.1 m
  Max:  2,961.7 m

Top 5 towns closest to a shopping mall:
town
CENTRAL AREA     315
CHOA CHU KANG    420
BUKIT PANJANG    477
SENGKANG         483
CLEMENTI         509

Top 5 towns furthest from a shopping mall:
town
GEYLANG             780
KALLANG/WHAMPOA     829
MARINE PARADE       856
JURONG EAST        1022
BEDOK              1151


## Phase 10 — Healthcare Distance (Polyclinics & Hospitals)

Load 38 healthcare facilities (27 polyclinics + 11 hospitals) from `data/healthcare_address.csv`. Geocode each by postal code via the OneMap API, cache to `data/healthcare_geocode_cache.json`, then compute the nearest distance and name for every flat.

In [24]:
HEALTHCARE_CSV           = '../../data/healthcare_address.csv'
HEALTHCARE_GEOCODE_CACHE = '../../data/healthcare_geocode_cache.json'

# Load healthcare facilities — drop trailing empty rows
healthcare_raw = pd.read_csv(HEALTHCARE_CSV).dropna(subset=['institution'])
print(f"Healthcare facilities loaded: {len(healthcare_raw)}")
print(healthcare_raw['institution'].tolist())

# Load or create geocode cache
if os.path.exists(HEALTHCARE_GEOCODE_CACHE):
    with open(HEALTHCARE_GEOCODE_CACHE) as f:
        hc_cache = json.load(f)
    print(f"\nLoaded existing cache: {len(hc_cache)} postal codes")
else:
    hc_cache = {}
    print("\nNo existing cache — starting fresh.")

new_calls = 0
for _, row in healthcare_raw.iterrows():
    postal = str(int(row['postal_code'])).zfill(6)
    if postal in hc_cache:
        continue
    try:
        resp = _session.get(
            GEOCODE_URL,
            params={'searchVal': postal, 'returnGeom': 'Y',
                    'getAddrDetails': 'Y', 'pageNum': 1},
            headers={'Authorization': f'Bearer {_get_token()}'},
            timeout=10
        )
        resp.raise_for_status()
        results = resp.json().get('results', [])
        if results:
            hc_cache[postal] = {'lat': float(results[0]['LATITUDE']),
                                'lon': float(results[0]['LONGITUDE'])}
        else:
            print(f"  WARNING: no results for {row['institution']} (postal {postal})")
            hc_cache[postal] = None
    except Exception as e:
        print(f"  ERROR geocoding {row['institution']}: {e}")
        hc_cache[postal] = None
    new_calls += 1
    time.sleep(0.3)

with open(HEALTHCARE_GEOCODE_CACHE, 'w') as f:
    json.dump(hc_cache, f)

ok  = sum(1 for v in hc_cache.values() if v is not None)
bad = sum(1 for v in hc_cache.values() if v is None)
print(f"\nGeocoding complete: {ok} succeeded, {bad} failed")
print(f"Cache saved to: {HEALTHCARE_GEOCODE_CACHE}")

# Build coordinate and name arrays for geocoded facilities
hc_rows = []
for _, row in healthcare_raw.iterrows():
    postal = str(int(row['postal_code'])).zfill(6)
    cached = hc_cache.get(postal)
    if cached is not None:
        hc_rows.append({'name': row['institution'],
                        'lat':  cached['lat'],
                        'lon':  cached['lon']})
hc_geocoded_df = pd.DataFrame(hc_rows)
hc_coords      = hc_geocoded_df[['lat', 'lon']].values.astype(float)
hc_names_arr   = hc_geocoded_df['name'].values
print(f"\nUsing {len(hc_coords)} geocoded healthcare facilities for distance computation.")

# Nearest healthcare distance and name
df['dist_nearest_healthcare_m'], nearest_hc = nearest_with_name_vectorized(
    df['lat'].values.astype(float),
    df['lon'].values.astype(float),
    hc_coords,
    hc_names_arr
)
df['nearest_healthcare_name'] = nearest_hc

non_null = df['dist_nearest_healthcare_m'].notna().sum()
null_cnt = df['dist_nearest_healthcare_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY — dist_nearest_healthcare_m")
print(f"{'='*60}")
print(f"Non-null: {non_null:,}   Null (failed geocode): {null_cnt:,}")
print(f"  Min:  {df['dist_nearest_healthcare_m'].min():,.1f} m")
print(f"  Mean: {df['dist_nearest_healthcare_m'].mean():,.1f} m")
print(f"  Max:  {df['dist_nearest_healthcare_m'].max():,.1f} m")

town_hc = (df.groupby('town')['dist_nearest_healthcare_m']
             .mean().sort_values()
             .rename('mean_dist_healthcare_m').round(0).astype(int))
print(f"\nTop 5 towns closest to a healthcare facility:")
print(town_hc.head(5).to_string())
print(f"\nTop 5 towns furthest from a healthcare facility:")
print(town_hc.tail(5).to_string())

Healthcare facilities loaded: 37
['Ang Mo Kio Polyclinic', 'Bedok Polyclinic', 'Bukit Batok Polyclinic', 'Bukit Merah Polyclinic', 'Choa Chu Kang Polyclinic', 'Clementi Polyclinic', 'Geylang Polyclinic', 'Hougang Polyclinic', 'Jurong Polyclinic', 'Marine Parade Polyclinic', 'Outram Polyclinic', 'Pasir Ris Polyclinic', 'Queenstown Polyclinic', 'Sengkang Polyclinic', 'Tampines Polyclinic', 'Toa Payoh Polyclinic', 'Woodlands Polyclinic', 'Yishun Polyclinic', 'Pioneer Polyclinic', 'Punggol Polyclinic', 'Eunos polyclinc', 'Sembawang Polyclinc', 'Kallang Polyclinic', 'Bukit Panjang Polyclinic', 'Khatib Polyclinic', 'Tampines North Polyclinic', 'ALEXANDRA HOSPITAL', 'CHANGI GENERAL HOSPITAL', 'INSTITUTE OF MENTAL HEALTH', 'KHOO TECK PUAT HOSPITAL', "KK WOMEN'S AND CHILDREN'S HOSPITAL", 'NATIONAL UNIVERSITY HOSPITAL', 'NG TENG FONG GENERAL HOSPITAL & JURONG COMMUNITY HOSPITAL', 'SENGKANG GENERAL HOSPITAL', 'SINGAPORE GENERAL HOSPITAL', 'TAN TOCK SENG HOSPITAL PTE LTD', 'WOODLANDS HOSPITAL']

L

In [25]:
# ── Final Save — Write Fully Enriched Dataset ────────────────────────────────
# Distance columns:
#   dist_nearest_hawker_m    — nearest open hawker centre at transaction month
#   dist_cbd_m               — straight-line distance to Raffles Place MRT (CBD proxy)
#   dist_nearest_primary_m   — nearest MOE primary school
#   dist_nearest_park_m      — nearest NParks managed green space
#   dist_nearest_sportsg_m   — nearest SportSG managed sport facility
#   dist_nearest_mall_m      — nearest shopping mall (221 malls, deduplicated)
#   dist_nearest_healthcare_m — nearest polyclinic or hospital (38 facilities)
#
# Name columns:
#   nearest_hawker_name      — name of the nearest hawker centre
#   nearest_sportsg_name     — name of the nearest SportSG facility
#   nearest_mall_name        — name of the nearest shopping mall
#   nearest_healthcare_name  — name of the nearest polyclinic or hospital
#
# Within-1km list columns (pipe-separated, empty string if none):
#   primary_schools_1km      — names of primary schools within 1 km
#   parks_1km                — names of NParks parks within 1 km

OUTPUT_FILE = '../../merged_data/hdb_with_amenities_macro.csv'
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df):,} rows to: {OUTPUT_FILE}\n")

dist_cols = [
    'dist_nearest_hawker_m',
    'dist_cbd_m',
    'dist_nearest_primary_m',
    'dist_nearest_park_m',
    'dist_nearest_sportsg_m',
    'dist_nearest_mall_m',
    'dist_nearest_healthcare_m',
]
summary = pd.DataFrame({
    'non_null': [df[c].notna().sum() for c in dist_cols],
    'null':     [df[c].isna().sum()  for c in dist_cols],
    'min_m':    [round(df[c].min(), 1) for c in dist_cols],
    'mean_m':   [round(df[c].mean(), 1) for c in dist_cols],
    'max_m':    [round(df[c].max(), 1) for c in dist_cols],
}, index=dist_cols)
print(summary.to_string())

Saved 140,537 rows to: ../../merged_data/hdb_with_amenities_macro.csv

                           non_null  null  min_m   mean_m    max_m
dist_nearest_hawker_m        140447    90    0.0    949.3   4889.0
dist_cbd_m                   140447    90  591.8  12533.2  20312.7
dist_nearest_primary_m       140447    90   43.8    426.6   3296.6
dist_nearest_park_m          140447    90   44.3    763.8   2217.4
dist_nearest_sportsg_m       140447    90   42.6   1152.1   4296.1
dist_nearest_mall_m          140447    90    5.9    641.1   2961.7
dist_nearest_healthcare_m    140447    90   44.8   1039.6   3869.9


## Final Step — Save

In [28]:
OUTPUT_2026    = '../../merged_data/[FINAL]hdb_with_amenities_macro_2026.csv'
OUTPUT_PRE2026 = '../../merged_data/[FINAL]hdb_with_amenities_macro_pre2026.csv'

df_2026    = df[df['year'] == 2026]
df_pre2026 = df[df['year'] != 2026]

df_2026.to_csv(OUTPUT_2026, index=False)
print(f"Saved {len(df_2026):,} rows to {OUTPUT_2026}")

df_pre2026.to_csv(OUTPUT_PRE2026, index=False)
print(f"Saved {len(df_pre2026):,} rows to {OUTPUT_PRE2026}")

Saved 6,058 rows to ../../merged_data/[FINAL]hdb_with_amenities_macro_2026.csv
Saved 134,479 rows to ../../merged_data/[FINAL]hdb_with_amenities_macro_pre2026.csv
